# AI Agents Memory & Planning

## Memory Types

Based on the cognitive science analogy (Weng, 2023):

| Type | Analogy | Implementation | Capacity |
|------|---------|---------------|----------|
| **Sensory** | Fleeting perception | Current input tokens | Very limited |
| **Short-term (working)** | Active thought | Context window | ~100K-200K tokens |
| **Long-term** | Persistent knowledge | External DB / vector store | Unlimited |
| **Episodic** | Past experiences | Conversation history store | Unlimited |
| **Semantic** | General world knowledge | LLM weights (implicit) | Fixed at training |

---

## Memory Implementations

### 1. Buffer Memory
Store all messages verbatim. Simple but hits context limits.

### 2. Summary Memory
Periodically summarize old messages:
$$\text{summary}_{new} = \text{LLM}(\text{summary}_{old} + \text{recent\_messages})$$

### 3. Window Memory
Keep only the last $k$ messages:
$$\text{context} = [m_{t-k}, m_{t-k+1}, ..., m_t]$$

### 4. Vector Store Memory
Embed messages → store → retrieve most relevant at query time:
$$\text{retrieved} = \text{top-k}_{\cos}(\text{embed}(q), \{\text{embed}(m_i)\})$$

### 5. Entity Memory
Extract and maintain a structured knowledge graph of entities mentioned.

---

## Planning Strategies

### Chain-of-Thought (CoT)
Linear reasoning chain: $s_1 \to s_2 \to ... \to s_n \to \text{answer}$

### Tree-of-Thoughts (ToT)
Branch and prune: explore multiple paths, backtrack from dead ends.

### MCTS (Monte Carlo Tree Search)
$$\text{UCT}(s,a) = Q(s,a) + C \sqrt{\frac{\ln N(s)}{N(s,a)}}$$

Where $Q$ is exploitation (average reward), second term is exploration.

### Plan-and-Solve
1. Devise a plan: decompose task into subtasks
2. Execute each subtask
3. Synthesize results

---

## Self-Reflection: Reflexion

After a failed attempt, the agent generates a verbal self-reflection:

$$\hat{a}_t = \text{LLM}(\text{trajectory}_{t-1}, r_{t-1}, \hat{a}_{t-1})$$

Where $r$ is the reward/feedback from the environment. The reflection is stored in memory and informs the next attempt.

```
Attempt 1: Failed searched wrong keyword
Reflection: "I should search for the exact paper title, not a paraphrase"
Attempt 2: Succeeds uses reflection as guidance
```

---

## MemGPT Architecture

MemGPT manages its own memory like an OS manages RAM:
- **In-context storage**: Active working memory (limited)
- **External storage**: Archival memory (unlimited)
- **Recall storage**: Conversation history

The agent can page data in/out of context as needed.

In [1]:
import os, json
from openai import OpenAI
from collections import deque

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ── Buffer Memory ─────────────────────────────────────────────────────────────
class BufferMemory:
    def __init__(self):
        self.messages = []
    
    def add(self, role: str, content: str):
        self.messages.append({"role": role, "content": content})
    
    def get(self) -> list:
        return self.messages.copy()
    
    def clear(self):
        self.messages = []


# ── Window Memory ─────────────────────────────────────────────────────────────
class WindowMemory:
    def __init__(self, k: int = 10):
        self.k = k
        self.window = deque(maxlen=k * 2)  # k turns = k*2 messages
    
    def add(self, role: str, content: str):
        self.window.append({"role": role, "content": content})
    
    def get(self) -> list:
        return list(self.window)


# ── Summary Memory ────────────────────────────────────────────────────────────
class SummaryMemory:
    def __init__(self, summary_threshold: int = 10):
        self.summary = ""
        self.recent = []
        self.threshold = summary_threshold
    
    def add(self, role: str, content: str):
        self.recent.append({"role": role, "content": content})
        if len(self.recent) >= self.threshold:
            self._summarize()
    
    def _summarize(self):
        messages_text = "\n".join([f"{m['role']}: {m['content']}" for m in self.recent])
        prompt = f"Previous summary: {self.summary}\n\nNew messages:\n{messages_text}\n\nUpdate the summary:"
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200
        )
        self.summary = response.choices[0].message.content
        self.recent = []  # Clear after summarizing
        print(f"Memory summarized: {self.summary[:100]}...")
    
    def get(self) -> list:
        messages = []
        if self.summary:
            messages.append({"role": "system", "content": f"Conversation summary: {self.summary}"})
        messages.extend(self.recent)
        return messages

# Test
mem = WindowMemory(k=3)
mem.add("user", "What is ML?")
mem.add("assistant", "Machine learning is...")
mem.add("user", "Tell me more")
mem.add("assistant", "Sure, let's dive deeper...")
mem.add("user", "What about deep learning?")

print("Window memory (last 3 turns):")
for m in mem.get():
    print(f"  {m['role']}: {m['content']}")

Window memory (last 3 turns):
  user: What is ML?
  assistant: Machine learning is...
  user: Tell me more
  assistant: Sure, let's dive deeper...
  user: What about deep learning?


In [2]:
# ── Vector Store Memory with FAISS ────────────────────────────────────────────
# pip install faiss-cpu sentence-transformers
import numpy as np

class VectorMemory:
    def __init__(self, embedding_dim: int = 384):
        try:
            import faiss
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer('all-MiniLM-L6-v2')
            self.index = faiss.IndexFlatIP(embedding_dim)  # Inner product (cosine if normalized)
            self.memories = []
            self.available = True
        except ImportError:
            self.available = False
            print("faiss/sentence-transformers not installed using mock")
    
    def add(self, text: str, metadata: dict = None):
        if not self.available:
            self.memories = getattr(self, 'memories', []) + [{'text': text, 'metadata': metadata}]
            return
        embedding = self.model.encode([text], normalize_embeddings=True)
        self.index.add(embedding)
        self.memories.append({'text': text, 'metadata': metadata or {}})
    
    def search(self, query: str, k: int = 3) -> list:
        if not self.available or not self.memories:
            return self.memories[:k]
        query_emb = self.model.encode([query], normalize_embeddings=True)
        scores, indices = self.index.search(query_emb, min(k, len(self.memories)))
        return [self.memories[i] for i in indices[0] if i >= 0]

vmem = VectorMemory()
vmem.add("User prefers Python over JavaScript", {"type": "preference"})
vmem.add("User is building a RAG chatbot", {"type": "project"})
vmem.add("User knows advanced ML", {"type": "expertise"})
vmem.add("User's name is Alex", {"type": "identity"})

results = vmem.search("What programming language does the user know?")
print("Relevant memories:")
for r in results:
    print(f"  - {r['text']} [{r['metadata']}]")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Relevant memories:
  - User prefers Python over JavaScript [{'type': 'preference'}]
  - User knows advanced ML [{'type': 'expertise'}]
  - User is building a RAG chatbot [{'type': 'project'}]


In [3]:
# ── Reflexion Pattern ─────────────────────────────────────────────────────────
class ReflexionAgent:
    def __init__(self, task: str, max_attempts: int = 3):
        self.task = task
        self.max_attempts = max_attempts
        self.reflections = []
    
    def attempt(self, attempt_num: int) -> str:
        reflection_context = ""
        if self.reflections:
            reflection_context = f"\n\nPast reflections:\n" + "\n".join(
                [f"{i+1}. {r}" for i, r in enumerate(self.reflections)]
            )
        
        prompt = f"""Task: {self.task}{reflection_context}
        
        Attempt {attempt_num}: Provide your best answer."""
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200
        )
        return response.choices[0].message.content
    
    def reflect(self, attempt: str, feedback: str) -> str:
        prompt = f"""You attempted: {self.task}
        Your answer: {attempt}
        Feedback: {feedback}
        
        Write a brief reflection on what went wrong and how to improve:"""
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150
        )
        return response.choices[0].message.content
    
    def run(self, evaluator_fn):
        for i in range(1, self.max_attempts + 1):
            print(f"\n=== Attempt {i} ===")
            attempt = self.attempt(i)
            print(f"Answer: {attempt[:150]}")
            
            success, feedback = evaluator_fn(attempt)
            if success:
                print("✅ Task completed successfully!")
                return attempt
            
            if i < self.max_attempts:
                reflection = self.reflect(attempt, feedback)
                self.reflections.append(reflection)
                print(f"Reflection: {reflection[:100]}")
        
        return "Max attempts reached"

# Example task with evaluator
agent = ReflexionAgent("Write a haiku about machine learning that mentions 'gradient'")

def haiku_evaluator(response: str) -> tuple:
    lines = [l.strip() for l in response.strip().split('\n') if l.strip()]
    has_gradient = 'gradient' in response.lower()
    is_haiku = len(lines) >= 3
    if has_gradient and is_haiku:
        return True, "Perfect!"
    if not has_gradient:
        return False, "Missing the word 'gradient'"
    return False, "Not a proper haiku (needs 3 lines)"

result = agent.run(haiku_evaluator)


=== Attempt 1 ===


## Additional Learning Resources

### Papers
- [Reflexion (Shinn et al., 2023)](https://arxiv.org/abs/2303.11366)
- [MemGPT (Packer et al., 2023)](https://arxiv.org/abs/2310.08560)
- [LLM+P: LLM + Planning (Liu et al., 2023)](https://arxiv.org/abs/2304.11477)
- [Tree of Thoughts (Yao et al., 2023)](https://arxiv.org/abs/2305.10601)
- [Self-Refine (Madaan et al., 2023)](https://arxiv.org/abs/2303.17651)

### Frameworks
- [LangChain Memory Docs](https://python.langchain.com/docs/concepts/memory/)
- [MemGPT / Letta](https://www.letta.com/)
- [Zep Memory (production memory layer)](https://www.getzep.com/)

### Blogs
- [LLM Powered Autonomous Agents Lilian Weng](https://lilianweng.github.io/posts/2023-06-23-llm-agent/)